# Saved-result paper analysis — CPU only

This notebook contains only the new analysis sections (24–33) from the extended experiment notebook. **Run this notebook top to bottom in a fresh kernel.** It reads your existing results and performs no model loading, training, teacher requests or evaluation inference. Check `PA_ROOT` in Section 24; the namespace and single seed match the supplied `tree.txt`.

With the artifacts listed in your tree, the analysis exports 27 CSV/LaTeX table pairs, 13 PNG/PDF figure pairs, posterior method draws, a results/discussion writing aid, provenance files and a ZIP. Optional diagnostic gaps are explicitly reported. Numeric outputs are intentionally empty in this delivered copy: run it on your saved results to populate them.


## 24. Setup: analysis only, no model imports or experiment calls

Run this section and every section below it in a fresh CPU kernel. **Do not use Run All on the full experiment notebook:** earlier cells can load models, call the teacher or run experiments. No definitions from earlier sections are needed. The default root, namespace, seed and baseline tag below match `tree.txt`; change the root if you moved the results. Dependencies: numpy, pandas, scipy, matplotlib, seaborn, pyarrow, jinja2, threadpoolctl. These are already in most working training environments; no torch or CUDA is needed.

All outputs go to a new `paper_analysis/<namespace>` folder. The original experiment results are only read. `PA_DRAWS=10000` is the reporting default. A lower value can be used to test execution, but use the stated final draw count for the paper. Repeated execution overwrites only this new analysis folder's named reports and ZIP.

In [1]:
from pathlib import Path
from itertools import combinations
import hashlib, json, math, re, unicodedata, zipfile
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy import stats
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import jinja2
try:
    from IPython.display import display
except ImportError:
    display = print

PA_ROOT = Path('/root/autodl-tmp/value_alignment_benchmark')
PA_NAMESPACE = 'paper_0726c6bf1530_7df3e726'  # Explicitly selected from tree.txt
PA_ACTIVE_TAG = '0726c6bf1530'
PA_SEEDS = (13,)
PA_DRAWS = 10000
PA_RANDOM_SEED = 20260909
PA_VERIFY_HASHES = True
PA_METHODS = ('sft','dpo','hypo','ipo','simpo','orpo','kto','caa_residual','caa_attention')
PA_LABELS = dict(zip(PA_METHODS, ('SFT','DPO','HyPO','IPO','SimPO','ORPO','KTO','CAA-R','CAA-A')))
PA_MAPPING = {'Self_direction_thought': 'Self_direction', 'Self_direction_action': 'Self_direction', 'Stimulation': 'Stimulation', 'Hedonism': 'Hedonism', 'Achievement': 'Achievement', 'Power_dominance': 'Power', 'Power_resources': 'Power', 'Face': 'Power', 'Security_personal': 'Security', 'Security_societal': 'Security', 'Conformity_rules': 'Conformity', 'Conformity_interpersonal': 'Conformity', 'Tradition': 'Tradition', 'Humility': 'Tradition', 'Benevolence_caring': 'Benevolence', 'Benevolence_dependability': 'Benevolence', 'Universalism_concern': 'Universalism', 'Universalism_nature': 'Universalism', 'Universalism_tolerance': 'Universalism', 'Universalism_objectivity': 'Universalism'}
PA_VALUES = tuple(dict.fromkeys(PA_MAPPING.values()))
PA_PROBS = {'kvs': [f'p_rating_{i}' for i in range(1,7)], 'aita':['p_NTA','p_NEUTRAL','p_YTA']}
PA_METRICS = ('target_drop','drift','selectivity','aita_primary','aita_strict')
PA_BLUE, PA_ORANGE = '#286A9B', '#BC713B'
PA_CMAP = LinearSegmentedColormap.from_list('pa_signed', [PA_BLUE,'#FAFAF8',PA_ORANGE])
PA_OUT = PA_ROOT / 'paper_analysis' / PA_NAMESPACE
PA_OUT.mkdir(parents=True, exist_ok=True)
PA_INPUTS, PA_FILES = {}, []
PA_LIMITATIONS = []
sns.set_theme(style='whitegrid', context='paper', font='DejaVu Sans')
plt.rcParams.update({'pdf.fonttype':42,'axes.spines.top':False,'axes.spines.right':False,'grid.alpha':.2})

def pa_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def pa_read(path):
    path = Path(path)
    if not path.is_file(): raise FileNotFoundError(f'Required saved artifact missing: {path}')
    PA_INPUTS[str(path)] = {'path':str(path), 'bytes':path.stat().st_size, 'sha256':pa_hash(path)}
    if path.suffix == '.parquet': return pd.read_parquet(path)
    if path.suffix == '.csv': return pd.read_csv(path)
    return json.loads(path.read_text())

def pa_table(frame, name):
    if frame.empty: raise ValueError(f'No data for {name}; refusing an empty paper table')
    for ext in ('csv','tex'):
        path = PA_OUT / f'{name}.{ext}'
        if ext == 'csv': frame.to_csv(path, index=False)
        else: path.write_text(frame.to_latex(index=False, escape=True, na_rep='--', float_format=lambda v:f'{v:.4f}'))
        PA_FILES.append(path)
    return frame

def pa_figure(fig, name, note=''):
    fig.text(.01,.008,f'Seed(s): {PA_SEEDS}. {note}',fontsize=8)
    fig.tight_layout(rect=(0,.055,1,.97))
    for ext in ('pdf','png'):
        path = PA_OUT / f'{name}.{ext}'
        fig.savefig(path,dpi=300,bbox_inches='tight'); PA_FILES.append(path)
    plt.close(fig)

def pa_id(method,target,seed): return f'{method}__{target}__seed{seed}'
def pa_raw_path(dataset,method,target,seed):
    return PA_ROOT / 'results' / 'raw' / PA_NAMESPACE / dataset / f'{pa_id(method,target,seed)}.parquet'

def pa_taxonomy(frame):
    expected = frame.refined_value.map(PA_MAPPING)
    if expected.isna().any(): raise ValueError('Unknown refined values: '+str(frame.loc[expected.isna(),'refined_value'].unique()))
    if 'basic_value' in frame and (frame.basic_value.notna() & frame.basic_value.ne(expected)).any():
        raise ValueError('Stored basic_value contradicts the registered taxonomy')
    return frame.assign(basic_value=expected)

def pa_probs(frame,columns):
    p=frame[columns].to_numpy(float)
    if not np.isfinite(p).all() or (p<0).any() or (p>1).any() or not np.allclose(p.sum(1),1,atol=1e-6):
        raise ValueError('Invalid saved probability distribution')
    return p

def pa_macro(frame,column): return frame.groupby('refined_value')[column].mean().mean()
def pa_normalize(text): return re.sub(r'\s+',' ',unicodedata.normalize('NFKC',str(text))).strip().lower()
print('Analysis source:', PA_ROOT / 'results' / 'raw' / PA_NAMESPACE)
print('New reports:', PA_OUT)



Analysis source: /root/autodl-tmp/value_alignment_benchmark/results/raw/paper_0726c6bf1530_7df3e726
New reports: /root/autodl-tmp/value_alignment_benchmark/paper_analysis/paper_0726c6bf1530_7df3e726


## 25. Validate complete predictions and build paired observations

**RQ1: What effects did each intervention have relative to its matched control?** Read all 99 result files for each dataset, verify their completion-file checksums, exact source coverage, taxonomy, metadata and probability distributions. Reconstruct effects from raw predictions, then cross-check against the existing method summary. This prevents stale or mixed aggregate files from silently entering the paper. AITA text is used locally to identify repeated posts across values; raw post text is not exported.

KVS selectivity = target rating drop − absolute non-target drift. AITA primary gain = −Δp(high) + Δp(low) + 0.5Δp(remaining). The strict gain drops the last term. Positive values follow the experiment's value-suppression convention; they are not ethical-quality or accuracy scores.

In [2]:
PA_KVS = pa_taxonomy(pa_read(PA_ROOT/'data'/'kvs_active_protocol_rows.parquet'))
PA_AITA = pa_taxonomy(pa_read(PA_ROOT/'data'/'aita_cleaned_evaluation.parquet'))
for frame in (PA_KVS, PA_AITA):
    if frame.source_id.isna().any() or frame.source_id.duplicated().any(): raise ValueError('Missing/duplicate source IDs in input data')
if 'post' not in PA_AITA or PA_AITA.post.isna().any() or PA_AITA.post.map(pa_normalize).eq('').any():
    raise ValueError('Cleaned AITA post text is required for clustering repeated posts')
PA_AITA['cluster_id'] = PA_AITA.post.map(lambda x: hashlib.sha256(pa_normalize(x).encode()).hexdigest())
PA_RAW = {}; audit=[]
for dataset in ('kvs','aita'):
    source = PA_KVS.loc[PA_KVS.split.eq('test')] if dataset=='kvs' else PA_AITA
    for method in PA_METHODS:
        for target in ('control',)+PA_VALUES:
            for seed in PA_SEEDS:
                path=pa_raw_path(dataset,method,target,seed); data=pa_taxonomy(pa_read(path)); done=pa_read(path.with_suffix('.DONE'))
                expected=source if dataset=='kvs' or target=='control' else source.loc[source.basic_value.eq(target)]
                if expected.empty: raise ValueError(f'No evaluation examples for {dataset}/{target}')
                if data.source_id.duplicated().any() or set(data.source_id)!=set(expected.source_id):
                    raise ValueError(f'Incorrect source coverage: {path.name}')
                data=data.set_index('source_id').loc[expected.source_id].reset_index()
                identity=['refined_value','basic_value']+(['high_value_stance','low_value_stance'] if dataset=='aita' else [])
                for col in identity:
                    if not np.array_equal(data[col].to_numpy(),expected[col].to_numpy()): raise ValueError(f'Label mismatch: {path}/{col}')
                for col,value in [('method',method),('target',target),('seed',seed)]:
                    if not data[col].eq(value).all(): raise ValueError(f'Wrong {col} in {path}')
                p=pa_probs(data,PA_PROBS[dataset])
                if dataset=='kvs':
                    variants=[pa_probs(data,[f'v{v}_p_{i}' for i in range(1,7)]) for v in range(1,4)]
                    if not np.allclose(p,np.mean(variants,axis=0),atol=1e-6): raise ValueError('Prompt mean differs from stored probabilities')
                    if not np.allclose(data.expected_rating,p@np.arange(1,7),atol=1e-6): raise ValueError('Invalid expected rating')
                if PA_VERIFY_HASHES and done.get('result_sha256')!=PA_INPUTS[str(path)]['sha256']: raise ValueError(f'Checksum mismatch: {path}')
                PA_RAW[(dataset,method,target,seed)]=data
                audit.append({'dataset':dataset,'method':method,'target':target,'seed':seed,'rows':len(data),'checks_passed':True})
pa_table(pd.DataFrame(audit),'prediction_coverage')
paired={'kvs':[],'aita':[]}; per_run=[]
for method in PA_METHODS:
    for target in PA_VALUES:
        for seed in PA_SEEDS:
            summaries={'method':method,'target':target,'seed':seed}
            for dataset in ('kvs','aita'):
                intervention=PA_RAW[(dataset,method,target,seed)].set_index('source_id')
                control=PA_RAW[(dataset,method,'control',seed)].set_index('source_id').loc[intervention.index]
                columns=['refined_value','basic_value']; frame=intervention[columns].reset_index()
                frame=frame.assign(method=method,target=target,seed=seed)
                if dataset=='kvs':
                    delta=intervention.expected_rating.to_numpy()-control.expected_rating.to_numpy()
                    frame['change']=delta;frame['target_drop']=-delta;frame['drift']=np.abs(delta);frame['cluster_id']=frame.source_id
                    mask=frame.basic_value.eq(target)
                    for metric,subset in [('target_drop',mask),('drift',~mask)]:
                        summaries[metric]=pa_macro(frame.loc[subset],metric)
                        summaries[metric+'_micro']=frame.loc[subset,metric].mean()
                    summaries['selectivity']=summaries['target_drop']-summaries['drift']
                    summaries['selectivity_micro']=summaries['target_drop_micro']-summaries['drift_micro']
                else:
                    if not intervention.high_value_stance.isin(['NTA','YTA','NEUTRAL']).all() or not intervention.low_value_stance.isin(['NTA','YTA','NEUTRAL']).all() or intervention.high_value_stance.eq(intervention.low_value_stance).any():
                        raise ValueError('Invalid AITA high/low stance mapping')
                    change=intervention[PA_PROBS['aita']].to_numpy()-control[PA_PROBS['aita']].to_numpy()
                    label_index={'NTA':0,'NEUTRAL':1,'YTA':2}; row=np.arange(len(frame))
                    high=intervention.high_value_stance.map(label_index).to_numpy();low=intervention.low_value_stance.map(label_index).to_numpy()
                    frame['high_component']=-change[row,high];frame['low_component']=change[row,low]
                    frame['remaining_component']=.5*change[row,3-high-low]
                    frame['aita_strict']=frame.high_component+frame.low_component
                    frame['aita_primary']=frame.aita_strict+frame.remaining_component
                    frame['cluster_id']=PA_AITA.set_index('source_id').loc[frame.source_id,'cluster_id'].to_numpy()
                    for metric in ('aita_primary','aita_strict'):
                        summaries[metric]=pa_macro(frame,metric);summaries[metric+'_micro']=frame[metric].mean()
                paired[dataset].append(frame)
            per_run.append(summaries)
PA_PAIRED={k:pd.concat(v,ignore_index=True) for k,v in paired.items()}
PA_PER_RUN=pa_table(pd.DataFrame(per_run),'per_target_seed_effects')
PA_TARGET=PA_PER_RUN.groupby(['method','target'],as_index=False).mean(numeric_only=True)
PA_SUMMARY=PA_TARGET.groupby('method',as_index=False)[list(PA_METRICS)].mean()
# Verify reconstructed point estimates against the existing aggregate if present.
old_path=PA_ROOT/'results'/'aggregate'/'method_summary.csv'
if old_path.exists():
    old=pa_read(old_path).set_index('method'); current=PA_SUMMARY.set_index('method')
    for new,oldcol in [('target_drop','target_drop_macro'),('drift','non_target_drift_macro'),('selectivity','selectivity_macro'),('aita_primary','aita_gain_macro'),('aita_strict','aita_strict_gain_macro')]:
        if set(old.index)!=set(current.index) or not np.allclose(current[new],old.loc[current.index,oldcol],atol=1e-8):
            raise ValueError('Reconstructed effects disagree with existing aggregates; check namespace/data provenance')
display(PA_SUMMARY.round(5))



,method,target_drop,drift,selectivity,aita_primary,aita_strict
0,caa_attention,0.17628,0.10051,0.07577,-0.02616,-0.01400
1,caa_residual,0.31200,0.23467,0.07733,-0.02675,-0.00268
2,dpo,0.00004,0.02195,-0.02190,0.00706,0.00416
3,hypo,0.00896,0.02143,-0.01247,0.00405,0.00264
4,ipo,0.01429,0.02522,-0.01092,0.00466,0.00257
5,kto,0.00025,0.01944,-0.01920,0.00185,0.00042
6,orpo,0.01496,0.02233,-0.00737,0.00178,0.00127
7,sft,1.02212,0.27966,0.74246,-0.01542,-0.00261
8,simpo,0.01074,0.02355,-0.01281,0.00723,0.00484


## 26. Paired cluster Bayesian bootstrap: fixed models and values

**RQ2: How uncertain are effects for these fitted models?** This section replaces the earlier statistical procedure with a paired **cluster Bayesian bootstrap**. Each observed cluster receives an independent Exp(1) weight; normalized weights have a Dirichlet(1,…,1) distribution. The same cluster weights are used for every method and target within a dataset. Source IDs define KVS clusters; normalized identical post texts define AITA clusters, including posts appearing in multiple value strata. Weighted example means are computed within each refined value, then the notebook's equal-refined-value and equal-target aggregation is retained.

Point estimates come from the original sample, not the average of bootstrap draws. The 2.5%–97.5% quantiles are **95% Bayesian-bootstrap intervals**, conditional on these fitted models and observed value strata. They exclude training-seed uncertainty, unseen target values and training/teacher-data uncertainty. The bootstrap assigns mass only to observed clusters. Missing AITA refined values remain absent; no synthetic coverage is introduced. This is an analysis-method change, so disclose it if the earlier procedure was preregistered.

Reference: [Rubin (1981), The Bayesian Bootstrap](https://doi.org/10.1214/aos/1176345338). Cluster weighting and the fixed-stratum aggregation are the explicit adaptation here.

In [3]:
# Exp(1) weights normalized across clusters are Dirichlet(1,...,1).
# Normalization cancels inside each weighted refined-value mean. A repeated
# AITA post gets one weight shared across every method, target and value stratum.
def pa_bootstrap(frame, dataset):
    keys=pd.MultiIndex.from_frame(frame[['method','target','seed']].drop_duplicates().sort_values(['method','target','seed']))
    clusters=pd.Index(sorted(frame.cluster_id.unique())); source_clusters=frame[['source_id','cluster_id']].drop_duplicates().set_index('source_id').cluster_id
    metric_names=('target_drop','drift') if dataset=='kvs' else ('aita_primary','aita_strict')
    prepared=[]; counts={m:np.zeros(len(keys)) for m in metric_names}
    points={m:np.zeros(len(keys)) for m in metric_names}
    for refined,group in frame.groupby('refined_value',sort=True):
        basic=group.basic_value.iloc[0]
        for metric in metric_names:
            selected=group.loc[group.target.eq(basic) if metric=='target_drop' else group.target.ne(basic)] if dataset=='kvs' else group
            if selected.empty: continue
            pivot=selected.pivot(index='source_id',columns=['method','target','seed'],values=metric).sort_index()
            if pivot.isna().any().any(): raise ValueError('Unpaired observations in bootstrap')
            positions=keys.get_indexer(pivot.columns); values=pivot.to_numpy(float)
            if not np.isfinite(values).all(): raise ValueError('Nonfinite effects')
            cluster_positions=clusters.get_indexer(source_clusters.loc[pivot.index])
            prepared.append((metric,positions,cluster_positions,values))
            counts[metric][positions]+=1;points[metric][positions]+=values.mean(0)
    draws={m:np.zeros((PA_DRAWS,len(keys))) for m in metric_names}
    rng=np.random.default_rng(PA_RANDOM_SEED+(0 if dataset=='kvs' else 1))
    with threadpool_limits(limits=1):
        for start in range(0,PA_DRAWS,128):
            end=min(start+128,PA_DRAWS);weights=rng.exponential(size=(end-start,len(clusters)))
            for metric,positions,ci,values in prepared:
                w=weights[:,ci];draws[metric][start:end,positions]+=(w@values)/w.sum(1)[:,None]
    for m in metric_names:
        if (counts[m]==0).any(): raise ValueError('An effect has no observed strata')
        points[m]/=counts[m];draws[m]/=counts[m]
    if dataset=='kvs':
        points['selectivity']=points['target_drop']-points['drift'];draws['selectivity']=draws['target_drop']-draws['drift']
    return keys,points,draws

def pa_interval(point,draws):
    low,high=np.quantile(draws,[.025,.975])
    return {'estimate':float(point),'lower95':float(low),'upper95':float(high)}
PA_METHOD_DRAWS={};intervals=[];target_intervals=[]
for dataset,frame in PA_PAIRED.items():
    print('Cluster Bayesian bootstrap:',dataset,PA_DRAWS,'draws',flush=True)
    keys,points,draws=pa_bootstrap(frame,dataset)
    for metric,values in draws.items():
        PA_METHOD_DRAWS[metric]=np.column_stack([values[:,keys.get_level_values('method')==m].mean(1) for m in PA_METHODS])
        for method in PA_METHODS:
            mask=keys.get_level_values('method')==method
            intervals.append({'method':method,'metric':metric,**pa_interval(points[metric][mask].mean(),values[:,mask].mean(1))})
            for target in PA_VALUES:
                select=mask & (keys.get_level_values('target')==target)
                target_intervals.append({'method':method,'target':target,'metric':metric,**pa_interval(points[metric][select].mean(),values[:,select].mean(1))})
PA_INTERVALS=pd.DataFrame(intervals); PA_TARGET_INTERVALS=pd.DataFrame(target_intervals)
for frame in (PA_INTERVALS,PA_TARGET_INTERVALS): frame['interval_type']='conditional_cluster_Bayesian_bootstrap_95pct'
for row in PA_INTERVALS.itertuples():
    if not np.isclose(row.estimate,PA_SUMMARY.set_index('method').loc[row.method,row.metric]): raise AssertionError('Bootstrap point mismatch')
pa_table(PA_INTERVALS,'method_intervals');pa_table(PA_TARGET_INTERVALS,'target_intervals')
draw_path=PA_OUT/'method_posterior_draws.npz'
np.savez_compressed(draw_path,methods=np.asarray(PA_METHODS),**PA_METHOD_DRAWS)
PA_FILES.append(draw_path)
main=pd.DataFrame({'method':[PA_LABELS[m] for m in PA_METHODS]})
for metric in PA_METRICS:
    frame=PA_INTERVALS.loc[PA_INTERVALS.metric.eq(metric)].set_index('method').loc[list(PA_METHODS)]
    main[metric+' [95% BB interval]']=[f'{r.estimate:.4f} [{r.lower95:.4f}, {r.upper95:.4f}]' for r in frame.itertuples()]
pa_table(main,'main_comparison');display(main)
fig,axes=plt.subplots(1,2,figsize=(11,4.8))
for ax,metric in zip(axes,('selectivity','aita_primary')):
    frame=PA_INTERVALS.loc[PA_INTERVALS.metric.eq(metric)].set_index('method').loc[list(PA_METHODS)];y=np.arange(len(frame))
    ax.hlines(y,frame.lower95,frame.upper95,color=PA_BLUE);ax.scatter(frame.estimate,y,color=PA_BLUE)
    ax.set_yticks(y,[PA_LABELS[m] for m in PA_METHODS]);ax.invert_yaxis();ax.axvline(0,color='black',lw=.6)
    ax.set(xlabel='KVS rating units' if metric=='selectivity' else 'AITA probability units',title=metric.replace('_',' ').title())
pa_figure(fig,'main_effect_intervals','95% Bayesian-bootstrap intervals; fixed fitted models and observed value strata.')



Cluster Bayesian bootstrap: kvs 10000 draws
Cluster Bayesian bootstrap: aita 10000 draws


,method,target_drop [95% BB interval],drift [95% BB interval],selectivity [95% BB interval],aita_primary [95% BB interval],aita_strict [95% BB interval]
0,SFT,"1.0221 [0.8932, 1.1540]","0.2797 [0.2611, 0.3006]","0.7425 [0.6129, 0.8771]","-0.0154 [-0.0251, -0.0057]","-0.0026 [-0.0107, 0.0055]"
1,DPO,"0.0000 [-0.0041, 0.0043]","0.0219 [0.0202, 0.0238]","-0.0219 [-0.0264, -0.0174]","0.0071 [0.0042, 0.0103]","0.0042 [0.0015, 0.0068]"
2,HyPO,"0.0090 [0.0042, 0.0137]","0.0214 [0.0197, 0.0232]","-0.0125 [-0.0170, -0.0081]","0.0041 [0.0007, 0.0073]","0.0026 [0.0000, 0.0053]"
3,IPO,"0.0143 [0.0102, 0.0184]","0.0252 [0.0228, 0.0277]","-0.0109 [-0.0147, -0.0074]","0.0047 [0.0016, 0.0077]","0.0026 [-0.0002, 0.0051]"
4,SimPO,"0.0107 [0.0057, 0.0161]","0.0236 [0.0214, 0.0259]","-0.0128 [-0.0174, -0.0081]","0.0072 [0.0041, 0.0104]","0.0048 [0.0022, 0.0075]"
5,ORPO,"0.0150 [0.0111, 0.0189]","0.0223 [0.0206, 0.0241]","-0.0074 [-0.0109, -0.0038]","0.0018 [-0.0013, 0.0047]","0.0013 [-0.0019, 0.0043]"
6,KTO,"0.0002 [-0.0037, 0.0043]","0.0194 [0.0175, 0.0215]","-0.0192 [-0.0236, -0.0150]","0.0018 [-0.0010, 0.0047]","0.0004 [-0.0025, 0.0032]"
7,CAA-R,"0.3120 [0.2728, 0.3493]","0.2347 [0.2168, 0.2516]","0.0773 [0.0435, 0.1107]","-0.0267 [-0.0466, -0.0035]","-0.0027 [-0.0189, 0.0151]"
8,CAA-A,"0.1763 [0.1344, 0.2148]","0.1005 [0.0908, 0.1113]","0.0758 [0.0349, 0.1130]","-0.0262 [-0.0411, -0.0112]","-0.0140 [-0.0270, -0.0015]"


## 27. Method comparisons and uncertainty in rankings

**RQ3: Are apparent method rankings stable under evaluation-data reweighting?** Export all 36 method-pair differences for each primary metric using shared draws, plus rank distributions and best-rank shares. Exact/numerical ties divide the best-rank share. These are descriptive Bayesian summaries; `BB_probability_a_gt_b` is not a frequentist p-value. No FDR or simultaneous-coverage claim is made. Avoid reporting only favorable pairwise comparisons.

In [4]:
pairwise=[];ranking=[]
for metric in ('selectivity','aita_primary'):
    samples=PA_METHOD_DRAWS[metric];points=PA_SUMMARY.set_index('method').loc[list(PA_METHODS),metric].to_numpy()
    ranks=stats.rankdata(-samples,axis=1,method='average')
    maxima=samples.max(1,keepdims=True);ties=np.isclose(samples,maxima,atol=1e-12,rtol=0)
    for i,m in enumerate(PA_METHODS):
        ranking.append({'method':m,'metric':metric,'observed_rank':stats.rankdata(-points,method='average')[i],
                        'mean_BB_rank':ranks[:,i].mean(),'rank_lower95':np.quantile(ranks[:,i],.025),'rank_upper95':np.quantile(ranks[:,i],.975),
                        'BB_best_share':(ties[:,i]/ties.sum(1)).mean()})
    for i,j in combinations(range(len(PA_METHODS)),2):
        diff=samples[:,i]-samples[:,j]
        pairwise.append({'method_a':PA_METHODS[i],'method_b':PA_METHODS[j],'metric':metric,
                         **pa_interval(points[i]-points[j],diff),'BB_probability_a_gt_b':np.mean(diff>0),
                         'interpretation':'descriptive Bayesian comparison; not a frequentist p-value or FDR claim'})
PA_PAIRWISE=pa_table(pd.DataFrame(pairwise),'paired_method_differences');PA_RANKS=pa_table(pd.DataFrame(ranking),'ranking_uncertainty')
fig,axes=plt.subplots(1,2,figsize=(11,4.8))
for ax,metric in zip(axes,('selectivity','aita_primary')):
    frame=PA_RANKS.loc[PA_RANKS.metric.eq(metric)].set_index('method').loc[list(PA_METHODS)]
    ax.barh([PA_LABELS[m] for m in PA_METHODS],frame.BB_best_share,color=PA_BLUE);ax.invert_yaxis();ax.set(xlim=(0,1),xlabel='Share of posterior draws ranked best',title=metric.replace('_',' ').title())
pa_figure(fig,'ranking_uncertainty','Ties split the best-rank share. Does not measure variability across training seeds.')



## 28. Value heterogeneity, omitted-value sensitivity and spillovers

**RQ4: Is the mean driven by a few values, and which other values move?** Heatmaps expose heterogeneous effects. Leave-one-value-out summaries reveal influential target values; they reuse the same trained models and are not cross-validation. Worst off-target effects complement average drift. The cross-value matrix uses signed intervention-minus-control rating changes; it is distinct from absolute-drift aggregation. Report sample coverage alongside extreme effects.

In [5]:
lovo=[]
for omitted in PA_VALUES:
    frame=PA_TARGET.loc[PA_TARGET.target.ne(omitted)].groupby('method')[['selectivity','aita_primary']].mean()
    for metric in frame:
        ranks=frame[metric].rank(ascending=False,method='average')
        for method in PA_METHODS: lovo.append({'omitted_value':omitted,'method':method,'metric':metric,'estimate':frame.loc[method,metric],'rank':ranks[method]})
PA_LOVO=pa_table(pd.DataFrame(lovo),'leave_one_value_out')
robust=PA_TARGET.groupby('method').agg(mean_selectivity=('selectivity','mean'),worst_value_selectivity=('selectivity','min'),
    positive_selectivity_values=('selectivity',lambda x:int((x>0).sum())),positive_AITA_values=('aita_primary',lambda x:int((x>0).sum())),worst_AITA_gain=('aita_primary','min')).reset_index()
pa_table(robust,'value_robustness')
cross=PA_PAIRED['kvs'].groupby(['method','target','seed','basic_value','refined_value'],as_index=False).change.mean()
cross=cross.groupby(['method','target','basic_value'],as_index=False).change.mean()
pa_table(cross,'cross_value_signed_changes')
spill=cross.loc[cross.target.ne(cross.basic_value)].copy();spill['absolute_change']=spill.change.abs()
worst=spill.sort_values('absolute_change',ascending=False).groupby(['method','target']).head(1)
pa_table(worst,'worst_off_target_value')
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
for ax,metric,title in zip(axes,('selectivity','aita_primary'),('KVS selectivity','AITA primary transfer')):
    matrix=PA_TARGET.pivot(index='method',columns='target',values=metric).reindex(index=PA_METHODS,columns=PA_VALUES)
    limit=max(np.abs(matrix.to_numpy()).max(),1e-8)
    sns.heatmap(matrix,cmap=PA_CMAP,vmin=-limit,vmax=limit,annot=True,fmt='.3f',ax=ax,yticklabels=[PA_LABELS[m] for m in PA_METHODS])
    ax.set(title=title,xlabel='Intervention target',ylabel='Method');ax.tick_params(axis='x',rotation=45);ax.tick_params(axis='y',rotation=0)
pa_figure(fig,'value_specific_effects','Equal-refined-value aggregation. Missing refined AITA strata are reported separately.')
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,metric in zip(axes,('selectivity','aita_primary')):
    matrix=PA_LOVO.loc[PA_LOVO.metric.eq(metric)].pivot(index='method',columns='omitted_value',values='rank').reindex(index=PA_METHODS,columns=PA_VALUES)
    sns.heatmap(matrix,cmap='Blues_r',vmin=1,vmax=9,annot=True,fmt='.1f',ax=ax,yticklabels=[PA_LABELS[m] for m in PA_METHODS],cbar_kws={'label':'Rank (1 best)'})
    ax.set(title=metric.replace('_',' ').title(),xlabel='Omitted target value',ylabel='Method');ax.tick_params(axis='x',rotation=45)
pa_figure(fig,'leave_one_value_out_ranks','Sensitivity across ten fixed values, not cross-validation or new model training.')
fig,axes=plt.subplots(3,3,figsize=(17,14));limit=max(cross.change.abs().max(),1e-8)
for ax,m in zip(axes.flat,PA_METHODS):
    matrix=cross.loc[cross.method.eq(m)].pivot(index='target',columns='basic_value',values='change').reindex(index=PA_VALUES,columns=PA_VALUES)
    sns.heatmap(matrix,cmap=PA_CMAP,vmin=-limit,vmax=limit,ax=ax,cbar=False)
    ax.set(title=PA_LABELS[m],xlabel='Measured value',ylabel='Target');ax.tick_params(axis='both',labelsize=7);ax.tick_params(axis='x',rotation=50);ax.tick_params(axis='y',rotation=0)
pa_figure(fig,'spillover_matrices',f'Intervention minus matched control. Shared range [{-limit:.3f}, {limit:.3f}] rating units; blue negative.')



## 29. Does intrinsic change transfer? Decompose the AITA score

**RQ5: Does KVS change generalize to AITA behavior?** Compare selectivity and AITA gain within each method across ten target values. This avoids conflating between-method and within-method association in one pooled correlation. Correlations and quadrant counts are descriptive with n=10; no causal mechanism is established. Decompose the external gain into reduced high-value stance probability, increased low-value stance probability and the remaining-label contribution. A primary/strict disagreement can then be explained quantitatively.

In [6]:
transfer=[]
for method,frame in PA_TARGET.groupby('method'):
    rho=stats.spearmanr(frame.selectivity,frame.aita_primary).statistic if frame.selectivity.nunique()>1 and frame.aita_primary.nunique()>1 else np.nan
    transfer.append({'method':method,'target_level_spearman_rho':rho,'n_target_values':len(frame),
                     'positive_on_both_values':int(((frame.selectivity>0)&(frame.aita_primary>0)).sum()),
                     'positive_KVS_negative_AITA_values':int(((frame.selectivity>0)&(frame.aita_primary<0)).sum())})
pa_table(pd.DataFrame(transfer),'intrinsic_external_transfer')
components=PA_PAIRED['aita'].groupby(['method','target','seed','refined_value'],as_index=False)[['high_component','low_component','remaining_component','aita_primary','aita_strict']].mean()
components=components.groupby(['method','target'],as_index=False).mean(numeric_only=True)
pa_table(components,'aita_stance_shift_decomposition')
fig,axes=plt.subplots(3,3,figsize=(11,10))
for ax,m in zip(axes.flat,PA_METHODS):
    frame=PA_TARGET.loc[PA_TARGET.method.eq(m)]
    ax.scatter(frame.selectivity,frame.aita_primary,color=PA_BLUE);ax.axhline(0,color='black',lw=.6);ax.axvline(0,color='black',lw=.6)
    ax.set(title=PA_LABELS[m],xlabel='KVS selectivity',ylabel='AITA primary gain')
pa_figure(fig,'intrinsic_external_transfer','Each point is a target value. Within-method correlations are descriptive (n=10).')
comp=components.groupby('method')[['high_component','low_component','remaining_component']].mean().reindex(PA_METHODS)
fig,ax=plt.subplots(figsize=(10,4.8));x=np.arange(len(PA_METHODS))
for j,(col,color) in enumerate(zip(comp.columns,[PA_BLUE,PA_ORANGE,'#555B65'])): ax.bar(x+(j-1)*.23,comp[col],width=.23,color=color,label=col.replace('_',' '))
ax.axhline(0,color='black',lw=.6);ax.set_xticks(x,[PA_LABELS[m] for m in PA_METHODS],rotation=30);ax.legend(frameon=False);ax.set(ylabel='Contribution to primary gain',title='Components of the AITA probability shift')
pa_figure(fig,'aita_gain_components','High: -delta p(high); low: +delta p(low); remaining: +0.5 delta p(remaining). Not accuracy.')



## 30. Prompt wording, aggregation sensitivity and ordinary control-training shift

**RQ6: Are conclusions sensitive to wording or aggregation?** Use the three already-scored prompts and macro/micro/strict definitions. Prompt variability does not measure independent training replication. A separate comparison measures how each method's control training shifted frozen Qwen before applying the targeted intervention. Do not substitute that comparison for intervention-versus-matched-control effects.

In [7]:
prompt=[];control_shift=[]
frozen=pa_taxonomy(pa_read(PA_ROOT/'baselines'/f'kvs_frozen_ratings_{PA_ACTIVE_TAG}.parquet'))
frozen=frozen.loc[frozen.split.eq('test')].set_index('source_id')
for method in PA_METHODS:
    for seed in PA_SEEDS:
        control=PA_RAW[('kvs',method,'control',seed)].set_index('source_id')
        if set(control.index)!=set(frozen.index): raise ValueError('Frozen baseline coverage mismatch')
        work=control[['refined_value','basic_value']].copy();work['change']=control.expected_rating-frozen.loc[control.index].expected_rating
        for basic,group in work.groupby('basic_value'): control_shift.append({'method':method,'seed':seed,'basic_value':basic,'control_minus_frozen':pa_macro(group,'change')})
        for target in PA_VALUES:
            frame=PA_RAW[('kvs',method,target,seed)].set_index('source_id');base=control.loc[frame.index]
            for v in (1,2,3):
                cols=[f'v{v}_p_{i}' for i in range(1,7)];delta=(base[cols].to_numpy()-frame[cols].to_numpy())@np.arange(1,7)
                work=frame[['refined_value','basic_value']].assign(drop=delta,drift=np.abs(delta));mask=work.basic_value.eq(target)
                prompt.append({'method':method,'target':target,'seed':seed,'prompt':v,'selectivity':pa_macro(work.loc[mask],'drop')-pa_macro(work.loc[~mask],'drift')})
PA_PROMPT=pa_table(pd.DataFrame(prompt),'prompt_specific_selectivity');pa_table(pd.DataFrame(control_shift),'control_training_shift_from_frozen')
variability=PA_PROMPT.groupby(['method','target'],as_index=False).agg(prompt_min=('selectivity','min'),prompt_max=('selectivity','max'),prompt_sd=('selectivity','std'))
variability['changes_sign_across_prompts']=(variability.prompt_min<0)&(variability.prompt_max>0)
pa_table(variability,'prompt_sensitivity_by_target')
pa_table(PA_PER_RUN[['method','target','seed','selectivity','selectivity_micro','aita_primary','aita_primary_micro','aita_strict','aita_strict_micro']],'metric_definition_sensitivity')
fig,ax=plt.subplots(figsize=(10,4.8))
for v,marker,color in [(1,'o',PA_BLUE),(2,'s',PA_ORANGE),(3,'^','#555B65')]:
    vals=PA_PROMPT.loc[PA_PROMPT.prompt.eq(v)].groupby('method').selectivity.mean().reindex(PA_METHODS)
    ax.scatter(np.arange(9)+(v-2)*.2,vals,marker=marker,color=color,label=f'Prompt {v}')
ax.axhline(0,color='black',lw=.6);ax.set_xticks(range(9),[PA_LABELS[m] for m in PA_METHODS],rotation=30);ax.legend(frameon=False);ax.set(ylabel='Mean selectivity',title='Sensitivity to KVS rating prompt wording')
pa_figure(fig,'prompt_sensitivity','Same saved predictions; no prompt-specific retraining.')
fig,ax=plt.subplots(figsize=(10,4.8));matrix=pd.DataFrame(control_shift).groupby(['method','basic_value']).control_minus_frozen.mean().unstack().reindex(index=PA_METHODS,columns=PA_VALUES)
sns.heatmap(matrix,cmap=PA_CMAP,center=0,ax=ax,yticklabels=[PA_LABELS[m] for m in PA_METHODS],cbar_kws={'label':'Rating change'})
ax.set(title='Matched control training shifts relative to frozen Qwen',xlabel='Measured value',ylabel='Method');ax.tick_params(axis='x',rotation=45)
pa_figure(fig,'control_vs_frozen','This is distinct from the targeted intervention-minus-control effect.')



## 31. Efficiency, Pareto trade-offs and learning-curve diagnostics

**RQ7: Which observed methods offer useful cost/effect trade-offs?** Join saved run manifests and learning curves. Cost comparisons average intervention targets only. A method is non-dominated if no other method has lower/equal recorded cost and higher/equal scores on both KVS selectivity and AITA gain, with at least one strict improvement. This Pareto classification is descriptive and does not incorporate score uncertainty. It is withheld if cost coverage is incomplete.

Times describe recorded preparation/training attempts, not full end-to-end study cost, and interrupted work may be omitted. Token budgets are nominal; they must not be used as measured throughput. Logged epochs/steps describe training progress, which may differ from the selected best checkpoint. Learning losses are separated by method because objective scales differ. Missing optional logs/manifests are recorded in the output notes rather than filled with zeros.

In [8]:
costs=[];curves=[]
for method in PA_METHODS:
    for target in ('control',)+PA_VALUES:
        for seed in PA_SEEDS:
            run_id=pa_id(method,target,seed); path=PA_ROOT/'manifests'/'runs'/PA_NAMESPACE/f'{run_id}.json'
            if not path.exists():
                PA_LIMITATIONS.append(f'Missing optional efficiency manifest: {run_id}');continue
            manifest=pa_read(path)
            row={'method':method,'target':target,'seed':seed,'run_id':run_id,'recorded_prepare_train_seconds':manifest.get('wall_clock_training_seconds',np.nan),
                 'peak_training_allocated_GiB':manifest.get('peak_gpu_memory_bytes',np.nan)/1024**3,
                 'trainable_parameters':manifest.get('trainable_parameters',np.nan),'nominal_tokens':manifest.get('tokens_processed',np.nan),
                 'registered_epochs':manifest.get('epochs',np.nan),'actual_epoch':manifest.get('actual_epoch',np.nan),'optimizer_steps':manifest.get('optimizer_steps',np.nan)}
            curve_path=PA_ROOT/'logs'/PA_NAMESPACE/f'{run_id}_training_curve.parquet'
            if curve_path.exists():
                curve=pa_read(curve_path).assign(method=method,target=target,seed=seed,run_id=run_id);curves.append(curve)
                if 'epoch' in curve: row['actual_epoch']=curve.epoch.max()
                if 'step' in curve: row['optimizer_steps']=curve.step.max()
            elif method in PA_METHODS[:7]: PA_LIMITATIONS.append(f'Missing training history: {run_id}')
            costs.append(row)
if costs:
    PA_COST=pa_table(pd.DataFrame(costs),'run_efficiency')
    # Target runs only: do not dilute intervention cost by including frozen controls.
    cost=PA_COST.loc[PA_COST.target.ne('control')].groupby('method',as_index=False).agg(
        recorded_prepare_train_seconds=('recorded_prepare_train_seconds','mean'),peak_training_allocated_GiB=('peak_training_allocated_GiB','max'),observed_target_runs=('run_id','size'))
    cost=cost.merge(PA_SUMMARY,on='method');cost['cost_coverage_complete']=cost.observed_target_runs.eq(len(PA_VALUES)*len(PA_SEEDS))
    cost['pareto_status']='unavailable_incomplete_cost_coverage'
    valid=cost.loc[cost.cost_coverage_complete & cost.recorded_prepare_train_seconds.notna()]
    if len(valid)==len(PA_METHODS):
        for i,row in valid.iterrows():
            weak=(valid.recorded_prepare_train_seconds<=row.recorded_prepare_train_seconds)&(valid.selectivity>=row.selectivity)&(valid.aita_primary>=row.aita_primary)
            strict=(valid.recorded_prepare_train_seconds<row.recorded_prepare_train_seconds)|(valid.selectivity>row.selectivity)|(valid.aita_primary>row.aita_primary)
            cost.loc[i,'pareto_status']='dominated' if (weak&strict).any() else 'non_dominated'
    pa_table(cost,'efficiency_pareto')
    fig,axes=plt.subplots(1,2,figsize=(11,4.8))
    for ax,metric in zip(axes,('selectivity','aita_primary')):
        for i,row in enumerate(cost.itertuples()):
            ax.scatter(row.recorded_prepare_train_seconds/60,getattr(row,metric),color=PA_BLUE)
            ax.annotate(str(i+1),(row.recorded_prepare_train_seconds/60,getattr(row,metric)),xytext=(3,3),textcoords='offset points',fontsize=8)
        ax.set(xlabel='Mean recorded intervention preparation/training (min)',ylabel=metric.replace('_',' '))
    key='; '.join(f'{i+1}: {PA_LABELS[r.method]}' for i,r in enumerate(cost.itertuples()))
    pa_figure(fig,'compute_effect_tradeoff',key+' — interrupted work may be unrecorded.')
else: PA_LIMITATIONS.append('Efficiency analysis unavailable: no manifests')
if curves:
    curves=pd.concat(curves,ignore_index=True);diagnostics=[]
    for run,group in curves.groupby('run_id'):
        record=group.iloc[0][['method','target','seed','run_id']].to_dict()
        for col in ('loss','eval_loss'):
            values=group.loc[group[col].notna()].sort_values('step') if col in group else pd.DataFrame()
            record[col+'_logged_points']=len(values)
            record[col+'_first']=values[col].iloc[0] if len(values) else np.nan
            record[col+'_last']=values[col].iloc[-1] if len(values) else np.nan
            record[col+'_best']=values[col].min() if len(values) else np.nan
        diagnostics.append(record)
    pa_table(pd.DataFrame(diagnostics),'training_diagnostics')
    for column in ('loss','eval_loss'):
        if column not in curves: PA_LIMITATIONS.append(f'Missing {column} curves');continue
        fig,axes=plt.subplots(3,3,figsize=(11,9))
        for ax,m in zip(axes.flat,PA_METHODS[:7]):
            frame=curves.loc[curves.method.eq(m)&curves[column].notna()]
            for _,run in frame.groupby('run_id'): ax.plot(run.step,run[column],alpha=.35,color=PA_BLUE,lw=.8)
            ax.set(title=PA_LABELS[m],xlabel='Optimizer step',ylabel=column)
        for ax in axes.flat[7:]: ax.set_visible(False)
        pa_figure(fig,'learning_curves_'+column,'Separate objective scales; individual runs, including matched controls. Lower loss is not cross-method superiority.')



## 32. Steering selection robustness and reference mismatch

**RQ8: Is CAA selection sensitive to the validation configuration?** Report the best-versus-runner-up validation margin and the number of grid settings within 0.01 rating units of the optimum. The threshold is a descriptive tolerance, not a significance test. These are validation selection diagnostics, not fresh test-set ablations. The train-only frozen reference mismatch table supports an exploratory HyPO–DPO comparison; it does not prove a mechanism.

In [9]:
selection_rows=[];grid_rows=[]
for kind in ('residual','attention'):
    for target in PA_VALUES:
        folder=PA_ROOT/'steering'/PA_NAMESPACE/'selection'/target/kind
        if not (folder/'selection_grid.csv').exists() or not (folder/'selected.json').exists():
            PA_LIMITATIONS.append(f'Missing steering selection artifacts: {kind}/{target}');continue
        grid=pa_read(folder/'selection_grid.csv');chosen=pa_read(folder/'selected.json')
        if grid[['layer','coefficient']].duplicated().any() or not np.isfinite(grid.selection_objective).all(): raise ValueError('Invalid steering grid')
        grid['kind']=kind;grid['target']=target;grid_rows.append(grid)
        vals=grid.selection_objective.sort_values(ascending=False).to_numpy()
        selected=grid.loc[grid.layer.eq(chosen['layer'])&np.isclose(grid.coefficient,chosen['coefficient'])]
        if len(selected)!=1 or not np.isclose(selected.selection_objective.iloc[0],vals[0]): raise ValueError('Selected CAA configuration is not a validation optimum')
        selection_rows.append({'kind':kind,'target':target,'selected_layer':chosen['layer'],'selected_coefficient':chosen['coefficient'],
            'best_validation_selectivity':vals[0],'best_minus_runner_up':vals[0]-vals[1] if len(vals)>1 else np.nan,
            'grid_configurations':len(grid),'within_0_01_of_best':int((vals>=vals[0]-.01).sum()),'median_grid_selectivity':np.median(vals)})
if selection_rows:
    pa_table(pd.DataFrame(selection_rows),'steering_selection_sensitivity');pa_table(pd.concat(grid_rows,ignore_index=True),'steering_validation_grid')
    fig,axes=plt.subplots(5,4,figsize=(14,13));limit=max(max(g.selection_objective.abs().max() for g in grid_rows),1e-8)
    for ax,grid in zip(axes.flat,grid_rows):
        sns.heatmap(grid.pivot(index='layer',columns='coefficient',values='selection_objective'),cmap=PA_CMAP,vmin=-limit,vmax=limit,annot=True,fmt='.2f',cbar=False,ax=ax)
        ax.set(title=f'{grid.kind.iloc[0]} / {grid.target.iloc[0]}',xlabel='Coefficient',ylabel='Layer')
    for ax in axes.flat[len(grid_rows):]:ax.set_visible(False)
    pa_figure(fig,'steering_validation_landscape','Validation-only selection, not a test-set ablation; common signed scale.')
margin_path=PA_ROOT/'baselines'/f'preference_reference_margins_{PA_ACTIVE_TAG}.parquet'
if margin_path.exists():
    margins=pa_taxonomy(pa_read(margin_path));margins=margins.loc[margins.split.eq('train')]
    mismatch=margins.assign(mismatch=margins.affirming_minus_opposing_margin.gt(0)).groupby('basic_value',as_index=False).agg(reference_mismatch_rate=('mismatch','mean'),train_n=('source_id','size')).rename(columns={'basic_value':'target'})
    difference=PA_TARGET.pivot(index='target',columns='method',values='selectivity')
    mismatch=mismatch.merge((difference.hypo-difference.dpo).rename('hypo_minus_dpo_selectivity').reset_index(),on='target')
    pa_table(mismatch,'hypo_dpo_reference_mismatch')
else: PA_LIMITATIONS.append('Reference-mismatch analysis unavailable: no frozen margins')



## 33. Data coverage, reproducibility bundle and evidence-grounded writing notes

**Paper reporting and reproducibility.** Export observed/missing refined-value cells, repeated-post cluster counts, available teacher audit flags, an exact input checksum inventory, analysis settings, all output checksums, and a ZIP bundle. `RESULTS_DISCUSSION_NOTES.md` names observed leaders and intrinsic/external disagreements using computed numbers and includes required limitations. Treat it as a writing aid, not a verified causal explanation or automatic significance statement. Missing optional diagnostics are explicitly listed.

The tree lists files, but does not contain their numerical contents. This extension was tested using synthetic fixtures; its real numerical outputs must be generated against your saved result directory. No synthetic findings are included in this notebook.

In [10]:
coverage=pd.DataFrame({'refined_value':list(PA_MAPPING)});coverage['basic_value']=coverage.refined_value.map(PA_MAPPING)
for split in ('train','eval','test'): coverage['kvs_'+split+'_n']=coverage.refined_value.map(PA_KVS.loc[PA_KVS.split.eq(split)].refined_value.value_counts()).fillna(0).astype(int)
coverage['aita_n']=coverage.refined_value.map(PA_AITA.refined_value.value_counts()).fillna(0).astype(int)
coverage['aita_missing']=coverage.aita_n.eq(0);coverage['aita_n_lt30']=coverage.aita_n.lt(30)
pa_table(coverage,'refined_value_coverage')
cluster_counts=PA_AITA.groupby('cluster_id').agg(rows=('source_id','size'),refined_values=('refined_value','nunique'))
cluster_summary=pd.DataFrame([{'aita_rows':len(PA_AITA),'unique_normalized_posts':len(cluster_counts),'posts_in_multiple_refined_values':int(cluster_counts.refined_values.gt(1).sum()),'rows_in_repeated_posts':int(cluster_counts.loc[cluster_counts.rows.gt(1),'rows'].sum())}])
pa_table(cluster_summary,'aita_post_cluster_audit')
cell_effects=PA_PAIRED['aita'].groupby(['method','target','seed','refined_value'],as_index=False).agg(aita_primary=('aita_primary','mean'),aita_strict=('aita_strict','mean'),n=('source_id','size'))
expected=pd.DataFrame([{'method':m,'target':b,'seed':s,'refined_value':r} for m in PA_METHODS for s in PA_SEEDS for r,b in PA_MAPPING.items()])
cell_effects=expected.merge(cell_effects,how='left',validate='one_to_one');cell_effects['n']=cell_effects.n.fillna(0).astype(int)
pa_table(cell_effects,'aita_refined_effects_with_missing_cells')
quality_path=PA_ROOT/'data'/f'teacher_quality_audit_{PA_ACTIVE_TAG}.parquet'
if quality_path.exists():
    quality=pa_read(quality_path);quality=quality.groupby('split',as_index=False).agg(records=('source_id','size'),unflagged=('quality_pass','sum'))
    quality['flagged_retained']=quality.records-quality.unflagged;pa_table(quality,'teacher_quality_by_split')
else: PA_LIMITATIONS.append('Teacher quality file unavailable')
notes=['# Results and discussion notes','', 'Generated from verified saved predictions. These are descriptive drafting aids, not automatic significance claims.','',
       '## Observed method effects']
for metric in ('selectivity','aita_primary'):
    best=PA_SUMMARY.sort_values(metric,ascending=False).iloc[0];iv=PA_INTERVALS.loc[PA_INTERVALS.method.eq(best.method)&PA_INTERVALS.metric.eq(metric)].iloc[0]
    notes.append(f'- Highest observed {metric}: {PA_LABELS[best.method]}, {best[metric]:.4f} [95% cluster Bayesian-bootstrap interval {iv.lower95:.4f}, {iv.upper95:.4f}]. This observed ranking does not establish a significant advantage over every other method.')
conflict=PA_SUMMARY.loc[(PA_SUMMARY.selectivity>0)&(PA_SUMMARY.aita_primary<0),'method'].tolist()
notes+=['','## Intrinsic–external disagreement',f'Positive mean KVS selectivity but negative mean AITA primary gain: {", ".join(PA_LABELS[m] for m in conflict) or "none"}. Inspect the strict metric and stance decomposition before attributing the difference to a mechanism.',
        '', '## Required qualifications',
        '- One seed: no training-seed variance or reproducibility claim. All fitted checkpoints and target values are held fixed.',
        '- KVS measures selective value suppression; AITA measures candidate-probability shifts, not accuracy or general ethical improvement.',
        '- Intervals use Dirichlet(1) cluster weights (implemented as shared exponential weights), with equal observed refined-value aggregation. KVS clusters are source IDs; AITA clusters are normalized post texts shared across values. The two datasets are reweighted independently.',
        '- The uncertainty model treats observed source/post clusters as exchangeable. Text clustering identifies normalized identical posts, not semantic near-duplicates.',
        '- These are Bayesian-bootstrap intervals and descriptive posterior ranking/comparison summaries. They are not frequentist p-values, family-wise simultaneous intervals, or FDR-adjusted tests. This replaces the earlier section 20 calculation; disclose an analysis change if preregistered.',
        '- Missing AITA refined-value cells stay missing; macro means cover observed strata only. Consult coverage and sample counts.',
        '- Leave-one-value-out and prompt/metric sensitivity reuse predictions; they are not additional training ablations or cross-validation.',
        '- Compute is recorded successful-attempt preparation/training time. Retries may be absent; nominal tokens are not measured throughput. Pareto status uses observed means and is descriptive.',
        '- Hyperparameter selection uses KVS validation only. The near-optimum tolerance 0.01 is descriptive and expressed in KVS rating units.',
        '- All-pairs exploration and mechanism correlations are post hoc. Report limitations and avoid selecting only favorable values/methods.',
        '', '## References', '[Rubin (1981), The Bayesian Bootstrap](https://doi.org/10.1214/aos/1176345338). The post-cluster weighting and fixed-stratum metric aggregation are the analysis choices implemented here.',
        '', '## Missing optional artifacts']+(['- '+s for s in sorted(set(PA_LIMITATIONS))] or ['None.'])
(PA_OUT/'RESULTS_DISCUSSION_NOTES.md').write_text('\n'.join(notes)+'\n');PA_FILES.append(PA_OUT/'RESULTS_DISCUSSION_NOTES.md')
config={'created_utc':datetime.now(timezone.utc).isoformat(),'root':str(PA_ROOT),'namespace':PA_NAMESPACE,'seeds':list(PA_SEEDS),'draws':PA_DRAWS,'analysis_seed':PA_RANDOM_SEED,'verify_result_hashes':PA_VERIFY_HASHES,'intervals':'95% conditional cluster Bayesian bootstrap','training_or_inference_performed':False,'optional_artifact_gaps':sorted(set(PA_LIMITATIONS))}
(PA_OUT/'analysis_configuration.json').write_text(json.dumps(config,indent=2));PA_FILES.append(PA_OUT/'analysis_configuration.json')
pd.DataFrame(PA_INPUTS.values()).to_csv(PA_OUT/'analysis_inputs.csv',index=False);PA_FILES.append(PA_OUT/'analysis_inputs.csv')
# Record the exact analysis source as well as the data hashes.
if 'PA_ANALYSIS_SOURCE' in globals():
    (PA_OUT/'analysis_source.py').write_text(PA_ANALYSIS_SOURCE);PA_FILES.append(PA_OUT/'analysis_source.py')
unique=sorted(set(PA_FILES));inventory=pd.DataFrame([{'file':p.name,'bytes':p.stat().st_size,'sha256':pa_hash(p)} for p in unique])
inventory.to_csv(PA_OUT/'analysis_output_inventory.csv',index=False)
archive_path=PA_ROOT/'paper_analysis'/f'{PA_NAMESPACE}_paper_analysis.zip'
archive_path.parent.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in unique+[PA_OUT/'analysis_output_inventory.csv']:archive.write(path,arcname=path.name)
print('Tables:',sum(p.suffix=='.csv' for p in unique)-1,'Figures:',sum(p.suffix=='.pdf' for p in unique))
print('Output directory:',PA_OUT);print('Archive:',archive_path)
display(inventory[['file','bytes']]);display(pd.DataFrame({'optional_artifact_gap':sorted(set(PA_LIMITATIONS))}))


Tables: 27 Figures: 13
Output directory: /root/autodl-tmp/value_alignment_benchmark/paper_analysis/paper_0726c6bf1530_7df3e726
Archive: /root/autodl-tmp/value_alignment_benchmark/paper_analysis/paper_0726c6bf1530_7df3e726_paper_analysis.zip


,file,bytes
0,RESULTS_DISCUSSION_NOTES.md,2752
1,aita_gain_components.pdf,15162
2,aita_gain_components.png,153840
3,aita_post_cluster_audit.csv,106
4,aita_post_cluster_audit.tex,200
...,...,...
79,value_robustness.tex,630
80,value_specific_effects.pdf,27270
81,value_specific_effects.png,571835
82,worst_off_target_value.csv,6262


,optional_artifact_gap
